---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 2.2: Add more tools

## 💬 Product Check-in: Code Quality

> **Sam (Product):** "Hey — a few users have mentioned that code examples from the agent don't always run. Syntax errors, missing indentation, that kind of thing. Not a huge volume but it's coming up enough to be worth addressing."
>
> **You:** "Yeah, the LLM is generating code from memory with nothing checking it before it goes to the user. It'll usually be close but occasionally has subtle errors."
>
> **Sam:** "Can we do anything about it?"
>
> **You:** "Easy fix with a tool call. The agent generates the code, validates it before responding, and self-corrects if it finds errors. Python has a built-in parser so there are no extra dependencies."
>
> **Sam:** "Great. Let's get that in."

---

Let's validate the problem exists first.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()

# Add judge model that's different
JUDGE_MODEL = "gemini-3-flash-preview"
MODEL = "gemini-3-flash-preview"
EXPERIMENT_NAME = os.environ.get("EXPERIMENT_2_NAME", "mlflow-agent-2")

if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.langchain.autolog()


prompt_version = mlflow.genai.load_prompt("prompts:/mlflow-agent-system/2")
SYSTEM_PROMPT = prompt_version.template

/Users/jon/Documents/GitHub/practical-agent-ops-mlflow3/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/04/15 10:27:53 INFO mlflow.tracking.fluent: Experiment with name 'mlflow-agent-2' does not exist. Creating a new experiment.
/Users/jon/Documents/GitHub/practical-agent-ops-mlflow3/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


MlflowException: RESOURCE_DOES_NOT_EXIST: Prompt with name=mlflow-agent-system not found

In [ ]:
# Create Agent

In [2]:
import ast

def validate_python_syntax(code: str) -> dict:
    """
    Validates Python syntax using the built-in ast module.
    No external dependencies required.

    Args:
        code: A string of Python code to validate.

    Returns:
        A dict with keys: valid, error, line, offset
    """
    try:
        ast.parse(code)
        return {
            "valid": True,
            "error": None,
            "line": None,
            "offset": None,
        }
    except SyntaxError as e:
        return {
            "valid": False,
            "error": str(e.msg),
            "line": e.lineno,
            "offset": e.offset,
        }

In [3]:
# Test tool
validate_python_syntax("def foo(x):\n    return x + 1")

{'valid': True, 'error': None, 'line': None, 'offset': None}

In [4]:
validate_python_syntax("public static void main(String[] args) {\n    System.out.println('Hello, world!');\n}")

{'valid': False, 'error': 'invalid syntax', 'line': 1, 'offset': 8}

In [ ]:
# Update Agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
import mlflow

mlflow.langchain.autolog()

llm = ChatGoogleGenerativeAI(
    model=MODEL,
)

tools = []  # add tools here when ready

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

def predict_fn(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content[0]["text"]

In [ ]:
code_quality_evals = [
    {
        # Only one right way to call this — specific args make the expected response tight
        "inputs": {"question": "Show me how to log a metric called 'accuracy' with a value of 0.95 at step 1 in MLflow."},
        "expectations": {
            "expected_response": "mlflow.log_metric('accuracy', 0.95, step=1)",
        },
    },
    {
        # Forces a specific parameter name and value — no ambiguity
        "inputs": {"question": "Show me how to log a parameter called 'learning_rate' with a value of 0.01 in MLflow."},
        "expectations": {
            "expected_response": "mlflow.log_param('learning_rate', 0.01)",
        },
    },
    {
        # Specific enough that the correct answer is a single line
        "inputs": {"question": "Show me how to set an experiment called 'my_experiment' in MLflow."},
        "expectations": {
            "expected_response": "mlflow.set_experiment('my_experiment')",
        },
    },
]
print(f"Code quality eval set size: {len(code_quality_evals)} examples")

# Add examples to full dataset